In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=1d9piyjBs2C12HcVJ7MyT4dufNv3jG&access_type=offline&code_challenge=rRRv97DIFTPCD_8y8tob1ZbRV6FfXOSAxMfab7qgRqU&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=schF2JdqnAQAz7tf7DDvvUusyTe3It&access_type=offline&code_challenge=80o14-feDQ6Y1Z1EDUUlCw1E0JphvsFZGRbSAmScaXA&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame
from gentropy.dataset.study_locus import StudyLocus
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics

from gentropy.susie_finemapper import SusieFineMapperStep

Loading BokehJS ...

In [2]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/23 21:51:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
path_to_release_folder="gs://open-targets-data-releases/25.03/"
#path_to_release_folder="gs://open-targets-pre-data-releases/24.12-uo_test-3/output/genetics/parquet/"
#path_to_release_folder="gs://ot_orchestration/releases/25.02_freeze1/"

si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

# MAKING EGL FROM EVIDENCE

In [10]:
evidence=session.spark.read.parquet("gs://open-targets-pre-data-releases/2503-testrun-1/output/evidence/")

In [6]:
dd= evidence.groupBy("datasourceId").count()
dd.show()

+------------------+-------+
|      datasourceId|  count|
+------------------+-------+
|               eva|3138382|
|              impc|1174471|
|gwas_credible_sets|1226520|
|            chembl| 573124|
|  expression_atlas| 229404|
|  uniprot_variants|  34560|
|       eva_somatic|   9719|
|  genomics_england|  34520|
|       gene_burden|  36123|
|cancer_gene_census|  82754|
|        slapenrich|  72406|
|          reactome|  10162|
|     crispr_screen|  21711|
|           intogen|   4359|
|    gene2phenotype|   4942|
| cancer_biomarkers|   1300|
|           clingen|   3003|
|            crispr|    517|
|            sysbio|    390|
|           progeny|    378|
+------------------+-------+



In [11]:
evidence = evidence.filter(f.col("score")>=0.95).filter(f.col("datasourceId").isin(["eva", "uniprot_variants", "gene2phenotype", "genomics_england", "clingen"])).cache()
evidence.count()


122889

In [12]:
dd= evidence.groupBy("datasourceId").count()
dd.show()

+----------------+-----+
|    datasourceId|count|
+----------------+-----+
|             eva|51900|
|uniprot_variants|33861|
|genomics_england|30950|
|  gene2phenotype| 4268|
|         clingen| 1910|
+----------------+-----+



In [13]:
evidence.show(1)

+------------+---------------+-------------+--------------------+------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+--------------------+---------------+--------------------+--------+-------------------+-------------------+----------------+--------------+-------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+------+------------+--------------------+-----------------+--------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+---------------+----------+------------+------------+

In [14]:
sgl0=evidence.select("diseaseId","targetId").distinct().cache()
sgl0.count()

12745

## CHEMBL

In [15]:
chembl=session.spark.read.parquet("gs://open-targets-pre-data-releases/2503-testrun-1/output/evidence/sourceId=chembl")

In [16]:
chembl.show(1)

+------------+---------------+-------------+-------------------+------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+-------------+------------+--------------------+-----------------+--------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+---------------+----------+------------+------------+-----------+----

In [17]:
chembl.printSchema()

root
 |-- datasourceId: string (nullable = true)
 |-- targetId: string (nullable = true)
 |-- alleleOrigins: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- allelicRequirements: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- altAminoAcid: string (nullable = true)
 |-- ancestry: string (nullable = true)
 |-- ancestryId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- betaConfidenceIntervalLower: double (nullable = true)
 |-- betaConfidenceIntervalUpper: double (nullable = true)
 |-- biologicalModelAllelicComposition: string (nullable = true)
 |-- biologicalModelGeneticBackground: string (nullable = true)
 |-- biologicalModelId: string (nullable = true)
 |-- biomarkerName: string (nullable = true)
 |-- biomarkers: struct (nullable = true)
 |    |-- geneExpression: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- name

In [18]:
clinical_phase_counts = chembl.groupBy("clinicalPhase").count()
clinical_phase_counts.show()

+-------------+------+
|clinicalPhase| count|
+-------------+------+
|          1.0|126491|
|          4.0|104823|
|          0.5|  7183|
|          3.0|122304|
|          2.0|212323|
+-------------+------+



In [19]:
chembl4=chembl.filter((f.col("clinicalPhase") == 4) | (f.col("clinicalPhase") == 3)) #&(f.col("clinicalStatus")=="Completed")
chembl4=chembl4.select("diseaseId","targetId","clinicalStatus","clinicalPhase","drugId").cache()
chembl4.count()

227127

In [20]:
clinical_phase_counts = chembl4.filter(f.col("clinicalPhase")==4).groupBy("clinicalStatus").count()
clinical_phase_counts.show()

+--------------------+-----+
|      clinicalStatus|count|
+--------------------+-----+
|           Suspended|   38|
|      Unknown status| 1819|
|           Completed|13846|
|                NULL|84311|
|Enrolling by invi...|  113|
|  Not yet recruiting|  581|
|          Recruiting| 1508|
|          Terminated| 1609|
|           Withdrawn|  548|
|Active, not recru...|  450|
+--------------------+-----+



In [21]:
sgl=chembl4.select("diseaseId","targetId").distinct().cache()
sgl.count()

25685

In [22]:
#sgl=session.spark.read.parquet("gs://genetics-portal-dev-analysis/xg1/Chembl_l2g_goldstandards/EGL_GSP_2409.parquet")
#sgl=session.spark.read.parquet("gs://genetics-portal-dev-analysis/xg1/Effector_gene_list.parquet")
#sgl=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/EGL_old_l2g_indirect_0.7")

#sgl=sgl.withColumnRenamed("efo_terms","diseaseId")
#sgl=sgl.withColumnRenamed("geneId","targetId")
sgl.count()

25685

## Old GS

In [25]:
sgl2=session.spark.read.json("gs://genetics_etl_python_playground/input/l2g/gold_standard/curation.json")

In [26]:
sgl2.printSchema()

root
 |-- association_info: struct (nullable = true)
 |    |-- ancestry: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- doi: string (nullable = true)
 |    |-- gwas_catalog_id: string (nullable = true)
 |    |-- neg_log_pval: double (nullable = true)
 |    |-- otg_id: string (nullable = true)
 |    |-- pubmed_id: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- gold_standard_info: struct (nullable = true)
 |    |-- evidence: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- class: string (nullable = true)
 |    |    |    |-- confidence: string (nullable = true)
 |    |    |    |-- curated_by: string (nullable = true)
 |    |    |    |-- description: string (nullable = true)
 |    |    |    |-- pubmed_id: string (nullable = true)
 |    |    |    |-- source: string (nullable = true)
 |    |-- gene_id: string (nullable = true)
 |    |-- highest_confidence: string (nullable = true)
 

In [27]:
sgl2.show(1,truncate=False)

+---------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------+---------------------------------------------------+-----------------------------------------------------------------------------+
|association_info                                               |gold_standard_info                                                                                                                                                                                                                                                                                           |metadata                                                      

In [28]:
sgl2.groupBy("gold_standard_info.highest_confidence").count().show()

+------------------+-----+
|highest_confidence|count|
+------------------+-----+
|              High|  853|
|               Low|  965|
|            Medium|  617|
+------------------+-----+



In [29]:
selected_df = sgl2.filter(f.col("gold_standard_info.highest_confidence").isin(["High","Medium"])).select( 
    f.col("gold_standard_info.gene_id").alias("geneId"),
    f.col("trait_info.ontology").alias("ontology"),
    f.col("trait_info.reported_trait_name").alias("diseaseFromSource"),
)

# Show the results
selected_df.show(1,truncate=False)

+---------------+-------------+------------------------------------------------+
|geneId         |ontology     |diseaseFromSource                               |
+---------------+-------------+------------------------------------------------+
|ENSG00000135697|[HMDB0000561]|Carotenoid and tocopherol levels (beta-carotene)|
+---------------+-------------+------------------------------------------------+
only showing top 1 row



In [30]:
exploded_df = selected_df.withColumn("efo_terms", f.explode("ontology")).drop("ontology")

exploded_df=exploded_df.withColumnRenamed("efo_terms","diseaseId").withColumnRenamed("geneId","targetId").select("diseaseId","targetId")
# Show the results
exploded_df.show(1,truncate=False)

+-----------+---------------+
|diseaseId  |targetId       |
+-----------+---------------+
|HMDB0000561|ENSG00000135697|
+-----------+---------------+
only showing top 1 row



## Combining

In [31]:
sgl.count()

25685

In [32]:
sgl.show(1)

+-----------+---------------+
|  diseaseId|       targetId|
+-----------+---------------+
|EFO_0003777|ENSG00000117601|
+-----------+---------------+
only showing top 1 row



In [33]:
exploded_df.count()

1491

In [34]:
exploded_df.show(1)

+-----------+---------------+
|  diseaseId|       targetId|
+-----------+---------------+
|HMDB0000561|ENSG00000135697|
+-----------+---------------+
only showing top 1 row



In [35]:
sgl0.count()

12745

In [36]:
sgl0.show(1)

+-------------+---------------+
|    diseaseId|       targetId|
+-------------+---------------+
|MONDO_0008116|ENSG00000258643|
+-------------+---------------+
only showing top 1 row



In [37]:
columns_order = exploded_df.columns

sgl_reordered = sgl.select(columns_order)
sgl0_reordered = sgl0.select(columns_order)

combined_df = exploded_df.union(sgl_reordered).union(sgl0_reordered)
combined_df=combined_df.cache()
# Show the results
combined_df.show(1,truncate=False)

+-----------+---------------+
|diseaseId  |targetId       |
+-----------+---------------+
|HMDB0000561|ENSG00000135697|
+-----------+---------------+
only showing top 1 row



In [38]:
combined_df.count()

39921

In [39]:
combined_df=combined_df.dropDuplicates(["targetId","diseaseId"]).cache()
combined_df.count()

38827

In [40]:
path_to_freeze="gs://open-targets-pre-data-releases/2503-testrun-1/output/"

fm=session.spark.read.parquet(path_to_freeze+"l2g_feature_matrix")
si=StudyIndex.from_parquet(session,path_to_freeze+"study")
cs=StudyLocus.from_parquet(session,path_to_freeze+"credible_set")

In [41]:
#fm=session.spark.read.parquet("gs://ot_orchestration/releases/24.10_freeze6/locus_to_gene_feature_matrix")
#si=StudyIndex.from_parquet(session,"gs://ot_orchestration/releases/24.10_freeze6/study_index")
#cs=StudyLocus.from_parquet(session,"gs://ot_orchestration/releases/24.10_freeze6/credible_set")

In [42]:
cs=cs.df.select("studyLocusId","studyId")
cs=cs.dropDuplicates(["studyLocusId"])
result_df = fm.join(cs, on="studyLocusId", how="left")

In [43]:
si=si.df.select("studyId","traitFromSource","traitFromSourceMappedIds").dropDuplicates(["studyId"])
result_df = result_df.join(si, on="studyId", how="left")

In [44]:
sgl=combined_df
sgl = sgl.withColumnRenamed("targetId", "geneId_sgl")

In [45]:
sgl.count()

38827

In [46]:
#joined_df = result_df.join(sgl, (f.array_contains(result_df["traitFromSourceMappedIds"], sgl["diseaseId"]))&(result_df["geneId"]==sgl["targetId"]),how="left")
joined_df = result_df.join(sgl, (f.array_contains(result_df["traitFromSourceMappedIds"], sgl["diseaseId"]))&(result_df["geneId"]==sgl["geneId_sgl"]),how="left")

In [ ]:
#joined_df.show(1)

+--------------------+--------------------+---------------+---------------------+---------------------+----------------------------------+-------------------------+--------------------------------------+-------------------+--------------------------------+---------------+----------------------------+--------------------+---------------------------------+------------------+-------------------------------+--------------+---------------+--------------------+---------------------------------+------------------+-------------------------------+---------------------+--------------------+---------------------------------+------------------+-------------------------------+----------+-----------------------+-------+--------------------+--------------------+------------------------+---------+----------+
|             studyId|        studyLocusId|         geneId|credibleSetConfidence|distanceFootprintMean|distanceFootprintMeanNeighbourhood|distanceSentinelFootprint|distanceSentinelFootprintNeighbo

In [48]:
#df=joined_df.filter(f.col("diseaseId").isNotNull())
df=joined_df.filter(f.col("diseaseId").isNotNull())
df=df.drop_duplicates(["studyLocusId","geneId","traitFromSourceMappedIds"])

In [49]:
df=df.cache()
df.count()

24240

In [50]:
df.drop_duplicates(["geneId","traitFromSourceMappedIds"]).count()

2906

In [51]:
folder_to_save="gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/"
suffix="2503-testrun-1-extended_EGL"
df.write.parquet(folder_to_save+suffix+"_prelim",mode="overwrite")

# STAGE 2

In [52]:
folder_to_save="gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/"
suffix="2503-testrun-1-extended_EGL"

df=session.spark.read.parquet(folder_to_save+suffix+"_prelim")

In [53]:
#df.printSchema()

In [54]:
path_to_freeze="gs://open-targets-pre-data-releases/2503-testrun-1/output/"

fm=session.spark.read.parquet(path_to_freeze+"l2g_feature_matrix")
si=StudyIndex.from_parquet(session,path_to_freeze+"study")
cs=StudyLocus.from_parquet(session,path_to_freeze+"credible_set")
cs=cs.df.select("studyLocusId","beta","pValueMantissa","pValueExponent","variantId")

In [55]:
df=df.join(cs, on="studyLocusId", how="left").cache()

In [56]:
df.count()

24240

In [57]:
df.printSchema()

root
 |-- studyLocusId: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- geneId: string (nullable = true)
 |-- credibleSetConfidence: float (nullable = true)
 |-- distanceFootprintMean: float (nullable = true)
 |-- distanceFootprintMeanNeighbourhood: float (nullable = true)
 |-- distanceSentinelFootprint: float (nullable = true)
 |-- distanceSentinelFootprintNeighbourhood: float (nullable = true)
 |-- distanceSentinelTss: float (nullable = true)
 |-- distanceSentinelTssNeighbourhood: float (nullable = true)
 |-- distanceTssMean: float (nullable = true)
 |-- distanceTssMeanNeighbourhood: float (nullable = true)
 |-- eQtlColocClppMaximum: float (nullable = true)
 |-- eQtlColocClppMaximumNeighbourhood: float (nullable = true)
 |-- eQtlColocH4Maximum: float (nullable = true)
 |-- eQtlColocH4MaximumNeighbourhood: float (nullable = true)
 |-- geneCount500kb: double (nullable = true)
 |-- isProteinCoding: float (nullable = true)
 |-- pQtlColocClppMaximum: float (nullable =

In [58]:
df.drop("traitFromSourceMappedIds").repartition(1).write.csv(folder_to_save+suffix+"_GSP_only.csv",header=True,mode="overwrite",sep="\t")

In [59]:
study_locus_ids = df.select("studyLocusId")

study_locus_id_list = [row.studyLocusId for row in study_locus_ids.collect()]


In [60]:
len(study_locus_id_list)

24240

In [61]:
fm1=fm.filter(f.col("studyLocusId").isin(study_locus_id_list))

In [62]:
fm1.count()

25/02/28 23:04:21 WARN DAGScheduler: Broadcasting large task binary with size 1575.4 KiB


1195392

In [63]:
cs=StudyLocus.from_parquet(session,path_to_freeze+"credible_set")
cs=cs.df.select("studyLocusId","beta","pValueMantissa","pValueExponent","variantId","studyId")

In [64]:
fm1=fm1.join(cs, on="studyLocusId", how="left").cache()

In [65]:
fm1.count()

25/02/28 23:07:28 WARN DAGScheduler: Broadcasting large task binary with size 1593.4 KiB
25/02/28 23:07:29 WARN DAGScheduler: Broadcasting large task binary with size 1577.6 KiB


1195392

In [ ]:
fm1.drop("traitFromSourceMappedIds").repartition(1).write.csv(folder_to_save+suffix+"_FM.csv",header=True,mode="overwrite",sep="\t")

25/03/01 01:29:11 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 475881 ms exceeds timeout 120000 ms
25/03/01 01:29:11 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/01 01:29:11 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

# STAGE 3 - IN R

# STAGE 4

In [3]:
path_to_read="gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/training_sets/"
path_to_save="gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/training_sets/"
suffix="patched_training_2503-testrun-1_all_string_005_extended_EGL_variants"

path_to_file=path_to_read+suffix+".tsv"
x=session.spark.read.csv(path_to_file,header=True,sep="\t")

In [4]:
#x.printSchema()

In [4]:
x=x.select("studyLocusId","geneId","GSP","efo_terms")

In [5]:
x.count()

80164

In [6]:
x.show(1)

+--------------------+---------------+---+-----------+
|        studyLocusId|         geneId|GSP|  efo_terms|
+--------------------+---------------+---+-----------+
|6edb472a3907a9a26...|ENSG00000147408|  0|EFO_0004530|
+--------------------+---------------+---+-----------+
only showing top 1 row



In [8]:
cs=sl.df.select("studyLocusId","variantId","studyId")

In [9]:
result_df = x.join(cs, on="studyLocusId", how="left")
result_df = result_df.cache()

In [10]:
result_df.count()

80164

In [11]:
#result_df = result_df.withColumn("sources", f.lit("manual_curation"))


In [12]:
result_df = result_df.withColumn(
    "goldStandardSet",
    f.when(f.col("GSP") == 1, "positive").otherwise("negative")
)



In [13]:
result_df.show(1)

+--------------------+---------------+---+-----------+---------------+------------+---------------+
|        studyLocusId|         geneId|GSP|  efo_terms|      variantId|     studyId|goldStandardSet|
+--------------------+---------------+---+-----------+---------------+------------+---------------+
|008b97f651f444888...|ENSG00000137845|  0|EFO_0004612|15_58429103_A_G|GCST90093014|       negative|
+--------------------+---------------+---+-----------+---------------+------------+---------------+
only showing top 1 row



In [14]:
#result_df.drop("GSP").write.parquet("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/training_set/training_v5.1_sum_col_string_filtered.parquet",mode="overwrite")

In [14]:
result_df=result_df.withColumnRenamed("efo_terms","traitFromSourceMappedId")

In [15]:
result_df.drop("GSP").write.json(path_to_save+suffix+".json",mode="overwrite")

In [16]:
result_df.count()

80164

In [17]:
path_to_save+suffix+".json"

'gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/training_sets/patched_training_2503-testrun-1_all_string_005_extended_EGL_variants.json'

# IGNORE EVERYTHING BELOW

In [5]:
x=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/training_set/v5_1_full.json")

In [6]:
x.printSchema()

root
 |-- geneId: string (nullable = true)
 |-- goldStandardSet: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- studyLocusId: string (nullable = true)
 |-- variantId: string (nullable = true)



In [7]:
x.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 10598|
|       negative|145702|
+---------------+------+



In [8]:
x.show(1)

+---------------+---------------+----------+--------------------+---------------+
|         geneId|goldStandardSet|   studyId|        studyLocusId|      variantId|
+---------------+---------------+----------+--------------------+---------------+
|ENSG00000131016|       negative|GCST003520|0b92f23c43ef385fb...|6_151627231_G_A|
+---------------+---------------+----------+--------------------+---------------+
only showing top 1 row



In [9]:
path_to_freeze="gs://open-targets-pre-data-releases/2503-testrun-1/output/"

si=StudyIndex.from_parquet(session,path_to_freeze+"study")

In [11]:
si.df.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- projectId: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- traitFromSource: string (nullable = true)
 |-- traitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- geneId: string (nullable = true)
 |-- biosampleFromSourceId: string (nullable = true)
 |-- biosampleId: string (nullable = true)
 |-- pubmedId: string (nullable = true)
 |-- publicationTitle: string (nullable = true)
 |-- publicationFirstAuthor: string (nullable = true)
 |-- publicationDate: string (nullable = true)
 |-- publicationJournal: string (nullable = true)
 |-- backgroundTraitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- backgroundDiseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- initialSampleSize: string (nullable = true)
 

In [12]:
result_df = x.join(si.df.select("studyId","traitFromSourceMappedIds"), on="studyId", how="left")
result_df = result_df.cache()

In [13]:
result_df.show(1)

+----------+---------------+---------------+--------------------+---------------+------------------------+
|   studyId|         geneId|goldStandardSet|        studyLocusId|      variantId|traitFromSourceMappedIds|
+----------+---------------+---------------+--------------------+---------------+------------------------+
|GCST003520|ENSG00000131016|       negative|0b92f23c43ef385fb...|6_151627231_G_A|           [EFO_0000305]|
+----------+---------------+---------------+--------------------+---------------+------------------------+
only showing top 1 row



In [14]:
result_df.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 10598|
|       negative|145702|
+---------------+------+



In [15]:
result_df = result_df.withColumn("efo_terms", f.element_at(f.col("traitFromSourceMappedIds"), 1))

In [16]:
result_df.show(1)

+----------+---------------+---------------+--------------------+---------------+------------------------+-----------+
|   studyId|         geneId|goldStandardSet|        studyLocusId|      variantId|traitFromSourceMappedIds|  efo_terms|
+----------+---------------+---------------+--------------------+---------------+------------------------+-----------+
|GCST003520|ENSG00000131016|       negative|0b92f23c43ef385fb...|6_151627231_G_A|           [EFO_0000305]|EFO_0000305|
+----------+---------------+---------------+--------------------+---------------+------------------------+-----------+
only showing top 1 row



In [17]:
result_df.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 10598|
|       negative|145702|
+---------------+------+



In [18]:
result_df = result_df.filter(f.col("efo_terms").isNotNull())

In [21]:
result_df.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 10143|
|       negative|140175|
+---------------+------+



In [20]:
result_df.drop("traitFromSourceMappedIds").write.json("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/training_set/v5_1_full_with_efo_no_explosion.json",mode="overwrite")

In [7]:
x_old=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/training_set/v5_1_full_with_efo_no_explosion.json")
x_old.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 10143|
|       negative|140175|
+---------------+------+



In [8]:
path_to_freeze="gs://open-targets-pre-data-releases/2503-testrun-1/output/"

fm=session.spark.read.parquet(path_to_freeze+"l2g_feature_matrix")
si=StudyIndex.from_parquet(session,path_to_freeze+"study")
cs=StudyLocus.from_parquet(session,path_to_freeze+"credible_set")
cs=cs.df.select("studyLocusId","beta","pValueMantissa","pValueExponent","variantId")

In [9]:
x_old.count()

150318

In [10]:
x_old.show(1)

+-----------+---------------+---------------+----------+--------------------+---------------+
|  efo_terms|         geneId|goldStandardSet|   studyId|        studyLocusId|      variantId|
+-----------+---------------+---------------+----------+--------------------+---------------+
|EFO_0000305|ENSG00000131016|       negative|GCST003520|0b92f23c43ef385fb...|6_151627231_G_A|
+-----------+---------------+---------------+----------+--------------------+---------------+
only showing top 1 row



In [13]:
fm.show(1)

+--------------------+---------------+---------------------+---------------------+----------------------------------+-------------------------+--------------------------------------+-------------------+--------------------------------+---------------+----------------------------+--------------------+---------------------------------+------------------+-------------------------------+--------------+---------------+--------------------+---------------------------------+------------------+-------------------------------+---------------------+--------------------+---------------------------------+------------------+-------------------------------+----------+-----------------------+-------+--------------------+
|        studyLocusId|         geneId|credibleSetConfidence|distanceFootprintMean|distanceFootprintMeanNeighbourhood|distanceSentinelFootprint|distanceSentinelFootprintNeighbourhood|distanceSentinelTss|distanceSentinelTssNeighbourhood|distanceTssMean|distanceTssMeanNeighbourhood|eQtl

In [14]:
x=x_old.join(fm, on=["studyLocusId", "geneId"], how="left").cache()

In [15]:
x.count()

150318

In [16]:
x.show(1)

+--------------------+---------------+-----------+---------------+------------+---------------+---------------------+---------------------+----------------------------------+-------------------------+--------------------------------------+-------------------+--------------------------------+---------------+----------------------------+--------------------+---------------------------------+------------------+-------------------------------+--------------+---------------+--------------------+---------------------------------+------------------+-------------------------------+---------------------+--------------------+---------------------------------+------------------+-------------------------------+----------+-----------------------+-------+--------------------+
|        studyLocusId|         geneId|  efo_terms|goldStandardSet|     studyId|      variantId|credibleSetConfidence|distanceFootprintMean|distanceFootprintMeanNeighbourhood|distanceSentinelFootprint|distanceSentinelFootprintNe

In [18]:
x.repartition(1).write.csv("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playgaround/training_set/v5_1_no_explosion_fm.csv",header=True,mode="overwrite",sep="\t")

In [ ]:
x_old=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playgaround/training_set/v5_1_full.json")
x_old.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 10598|
|       negative|145702|
+---------------+------+



In [72]:
x1=x.select("studyLocusId").distinct()
x1.count()

10281

In [73]:
x2=x_old.select("studyLocusId").distinct()
x2.count()

10277

In [74]:
# Find overlapping elements (common elements)
overlapping_elements = x1.join(x2, on="studyLocusId", how="inner")

# Find unique elements in x1
unique_in_x1 = x1.join(x2, on="studyLocusId", how="left_anti")

# Find unique elements in x2
unique_in_x2 = x2.join(x1, on="studyLocusId", how="left_anti")

# Show results
print("Overlapping elements:")
overlapping_elements.show()

print("Unique elements in x1:")
unique_in_x1.show()

print("Unique elements in x2:")
unique_in_x2.show()

Overlapping elements:


+--------------------+
|        studyLocusId|
+--------------------+
|045206a5e8948e613...|
|04c737b890f132dc7...|
|1237d52c16cdc4edc...|
|17669640cda402fcc...|
|19ef96a129e8e15f5...|
|1a020b0bd4564d4a0...|
|1ff47fc75659266d2...|
|24c1c261eb8280fff...|
|268f7b64001769c31...|
|29565f82e44f34fc3...|
|2b47a9bec15f965d3...|
|2cf94789d647b2fc3...|
|2f53355974d5b2d74...|
|31a153c006d98bdad...|
|3af4fe661109e39b9...|
|4295f455456802986...|
|461536b334974872e...|
|4b67cbc4669da923a...|
|4d5757cc8e2cc93be...|
|531443bb40ee475b3...|
+--------------------+
only showing top 20 rows

Unique elements in x1:


+--------------------+
|        studyLocusId|
+--------------------+
|5b42fc7248124b30c...|
|01328eb32db67229b...|
|078b7af52f5fee8c8...|
|12218ae1d08cdbad6...|
|96232b668e7c0ea6c...|
|e8fd2b54a287fc58e...|
|22f750b42cbd6312f...|
|5438ac521754c7715...|
|f3f06f4abaa9d5191...|
|d3fd2aa02f66477c2...|
+--------------------+

Unique elements in x2:


+--------------------+
|        studyLocusId|
+--------------------+
|0dfb15cea58c792d3...|
|f23f81eae59a117d8...|
|0e7d2bd84fcdd148a...|
|2e83f8edca240f34f...|
|2ee7338c95fcc1685...|
|c43b4570eaa27072d...|
+--------------------+



In [ ]:
unique_in_x2.count()

6

25/02/12 23:30:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 2012465 ms exceeds timeout 120000 ms
25/02/12 23:30:44 WARN SparkContext: Killing executors is not supported by current scheduler.
25/02/12 23:30:46 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

In [161]:
x.groupBy("goldStandardSet").count().show()

+---------------+------+
|goldStandardSet| count|
+---------------+------+
|       positive| 11313|
|       negative|154741|
+---------------+------+



In [67]:
x.filter(f.col("goldStandardSet")=="positive").select("traitFromSourceMappedId","geneId").distinct().count()

482

In [3]:
x=session.spark.read.json("gs://genetics_etl_python_playground/input/l2g/gold_standard/curation.json")

In [4]:
x.count()

2435

In [5]:
x.printSchema()

root
 |-- association_info: struct (nullable = true)
 |    |-- ancestry: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- doi: string (nullable = true)
 |    |-- gwas_catalog_id: string (nullable = true)
 |    |-- neg_log_pval: double (nullable = true)
 |    |-- otg_id: string (nullable = true)
 |    |-- pubmed_id: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- gold_standard_info: struct (nullable = true)
 |    |-- evidence: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- class: string (nullable = true)
 |    |    |    |-- confidence: string (nullable = true)
 |    |    |    |-- curated_by: string (nullable = true)
 |    |    |    |-- description: string (nullable = true)
 |    |    |    |-- pubmed_id: string (nullable = true)
 |    |    |    |-- source: string (nullable = true)
 |    |-- gene_id: string (nullable = true)
 |    |-- highest_confidence: string (nullable = true)
 

In [7]:
x.show(1,truncate=False)

+---------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------+---------------------------------------------------+-----------------------------------------------------------------------------+
|association_info                                               |gold_standard_info                                                                                                                                                                                                                                                                                           |metadata                                                      

In [8]:
x=session.spark.read.parquet("gs://genetics_etl_python_playground/static_assets/interaction")

In [10]:
x.count()

14062154

In [11]:
x.show(1)

+--------------+---------------+---------------+------------------+---------------+---------------+------------------+--------------------+--------------------+-----+-------+
|sourceDatabase|        targetA|           intA|intABiologicalRole|        targetB|           intB|intBBiologicalRole|            speciesA|            speciesB|count|scoring|
+--------------+---------------+---------------+------------------+---------------+---------------+------------------+--------------------+--------------------+-----+-------+
|        string|ENSG00000004059|ENSP00000000233|  unspecified role|ENSG00000215375|ENSP00000383023|  unspecified role|{human, Homo sapi...|{human, Homo sapi...|    2|  0.153|
+--------------+---------------+---------------+------------------+---------------+---------------+------------------+--------------------+--------------------+-----+-------+
only showing top 1 row



In [12]:
x=x.filter(f.col("sourceDatabase")=="string")

In [13]:
x.count()

13228819

In [14]:
x=x.filter(f.col("scoring")>=0.8)

In [15]:
x.count()

319762

In [17]:
x.drop("speciesA","speciesB").repartition(1).write.csv("gs://genetics-portal-dev-analysis/yt4/20241024_EGL_playground/training_set/interactions.tsv",mode="overwrite",header=True,sep="\t")